<div style="background-color:#0a0a0a;color:#00ff41;padding:20px 30px;font-family:'Courier New',Courier,monospace;border:1px solid #00ff41;border-radius:5px;box-shadow:inset 0 0 10px #000000,0 0 10px rgba(0,255,65,0.2);">
    <h1 style="color:#00ff41;margin:0 0 10px 0;font-weight:bold;text-transform:uppercase;">&gt; Práctica 04 - Redes Bayesianas y Naive Bayes_ &#9608;</h1>
</div>

**Asignatura:** Inteligencia Artificial / Machine Learning  
**Título de la práctica:** Aplicaciones de las Redes Bayesianas como herramientas de soporte a la toma de decisiones.  
**Objetivo:** Desarrollar modelos para realizar la clasificación de patrones mediante machine learning.  
**Dataset nuevo usado:** `data/breast_cancer_wisconsin_diagnostic.csv`  
**Enfoque del trabajo:** primero se analiza una Red Bayesiana clásica para comprender el razonamiento probabilístico; después se prepara un dataset real y se entrena un modelo Naive Bayes para clasificación.


<div style="background-color:#0a0a0a;color:#00ff41;padding:10px 15px;font-family:'Courier New',Courier,monospace;border-left:5px solid #00ff41;border-bottom:1px solid #00ff41;box-shadow:inset 0 0 5px #000000;">
    <h2 style="margin:0;text-transform:uppercase;">&gt; 1. Investigación: ejemplo de aplicación de una Red Bayesiana_ &#9608;</h2>
</div>

Antes de construir el modelo Naive Bayes, se revisa un ejemplo de Red Bayesiana para comprender cómo se representan variables, dependencias y probabilidades condicionales.

**Caso de estudio:** sistema de alarma domiciliaria.

Una alarma puede activarse por dos causas principales: un robo o un terremoto. Si la alarma suena, dos vecinos pueden llamar para avisar: John y María. El objetivo del caso es estimar la probabilidad de que exista un robo a partir de evidencias indirectas, como las llamadas de los vecinos.

Este caso es útil porque muestra una situación realista de incertidumbre: escuchar una llamada no confirma directamente un robo, pero sí modifica la probabilidad de que haya ocurrido. Por eso, la Red Bayesiana funciona como una herramienta de soporte a la toma de decisiones, ya que permite razonar con evidencia incompleta y actualizar conclusiones de forma ordenada.

En palabras simples, la red responde preguntas como: *si ocurrió cierta evidencia, qué tan probable es la causa que estoy investigando*.


<div style="background-color:#0a0a0a;color:#00ff41;padding:10px 15px;font-family:'Courier New',Courier,monospace;border-left:5px solid #00ff41;border-bottom:1px solid #00ff41;box-shadow:inset 0 0 5px #000000;">
    <h3 style="margin:0;text-transform:uppercase;">&gt; 1.a Revisión teórica breve_ &#9608;</h3>
</div>

El teorema de Bayes permite actualizar una probabilidad inicial cuando aparece nueva evidencia. Su forma general es:

$$P(H|E)=\frac{P(E|H)P(H)}{P(E)}$$

Donde:

- `H` es una hipótesis, por ejemplo: existe un robo.
- `E` es una evidencia, por ejemplo: John y María llaman.
- `P(H|E)` es la probabilidad posterior, es decir, la probabilidad actualizada después de observar la evidencia.

Una **Red Bayesiana** representa estas relaciones mediante nodos y conexiones dirigidas. Cada nodo posee una tabla de probabilidades que indica cómo cambia su comportamiento según sus causas o variables padre.

La idea central es combinar dos elementos: lo que se sabía antes de observar la evidencia y lo que aporta la nueva evidencia. Así se obtiene una probabilidad actualizada, más útil para decidir.


<div style="background-color:#0a0a0a;color:#00ff41;padding:10px 15px;font-family:'Courier New',Courier,monospace;border-left:5px solid #00ff41;border-bottom:1px solid #00ff41;box-shadow:inset 0 0 5px #000000;">
    <h3 style="margin:0;text-transform:uppercase;">&gt; 1.b Variables, valores, diagrama y conexiones_ &#9608;</h3>
</div>

| Variable | Descripción | Valores |
|---|---|---|
| B | Robo | Sí / No |
| E | Terremoto | Sí / No |
| A | Alarma | Sí / No |
| J | John llama | Sí / No |
| M | María llama | Sí / No |

**Diagrama de la Red Bayesiana:**

![Diagrama profesional de la Red Bayesiana de alarma](../img/red_bayesiana_alarma.svg)

| Conexión | Interpretación |
|---|---|
| `B -> A` | Un robo puede activar la alarma. |
| `E -> A` | Un terremoto puede activar la alarma. |
| `A -> J` | John llama dependiendo de si escucha la alarma. |
| `A -> M` | María llama dependiendo de si escucha la alarma. |

La estructura indica que `B` y `E` son causas posibles de `A`. A su vez, `J` y `M` no dependen directamente del robo o del terremoto, sino de si la alarma se activa. Esta separación permite simplificar el cálculo probabilístico.

Forma sencilla de leer el diagrama: primero pueden ocurrir las causas (`B` o `E`), luego puede sonar la alarma (`A`) y finalmente pueden aparecer las llamadas (`J` y `M`).


<div style="background-color:#0a0a0a;color:#00ff41;padding:10px 15px;font-family:'Courier New',Courier,monospace;border-left:5px solid #00ff41;border-bottom:1px solid #00ff41;box-shadow:inset 0 0 5px #000000;">
    <h3 style="margin:0;text-transform:uppercase;">&gt; 1.c Tablas con valores de probabilidad_ &#9608;</h3>
</div>

Las tablas de probabilidad se guardaron como archivos CSV en la carpeta `data/` para que la práctica sea reproducible y verificable. En esta sección se presentan las probabilidades iniciales y las tablas condicionales usadas por la red.

Estas tablas no son el dataset del modelo Naive Bayes; son los valores que permiten resolver manualmente las consultas de la Red Bayesiana del ejemplo.


In [1]:
from pathlib import Path
from itertools import product

import pandas as pd
from IPython.display import display

DATA_DIR = Path('../data') if Path('../data').exists() else Path('data')
TARGET_NAMES = ['malignant', 'benign']

prior_df = pd.read_csv(DATA_DIR / 'red_bayesiana_alarma_prior.csv')
alarm_df = pd.read_csv(DATA_DIR / 'red_bayesiana_alarma_cpt_alarma.csv')
calls_df = pd.read_csv(DATA_DIR / 'red_bayesiana_alarma_cpt_llamadas.csv')

print('Probabilidades iniciales')
display(prior_df)

print('CPT de la alarma: P(A|B,E)')
display(alarm_df)

print('CPT de llamadas: P(J|A) y P(M|A)')
display(calls_df)


Probabilidades iniciales


,variable,P_Si,P_No
0,Robo B,0.001,0.999
1,Terremoto E,0.002,0.998


CPT de la alarma: P(A|B,E)


,Robo,Terremoto,P_Alarma_Si,P_Alarma_No
0,Si,Si,0.950,0.050
1,Si,No,0.940,0.060
2,No,Si,0.290,0.710
3,No,No,0.001,0.999


CPT de llamadas: P(J|A) y P(M|A)


,Alarma,P_John_llama_Si,P_Maria_llama_Si
0,Si,0.90,0.70
1,No,0.05,0.01


<div style="background-color:#0a0a0a;color:#00ff41;padding:10px 15px;font-family:'Courier New',Courier,monospace;border-left:5px solid #00ff41;border-bottom:1px solid #00ff41;box-shadow:inset 0 0 5px #000000;">
    <h3 style="margin:0;text-transform:uppercase;">&gt; 1.d Dos ejemplos de predicción siguiendo la red_ &#9608;</h3>
</div>

La red factoriza la probabilidad conjunta así:

$$P(B,E,A,J,M)=P(B)P(E)P(A|B,E)P(J|A)P(M|A)$$

Se realizan dos consultas para demostrar cómo cambia la probabilidad de robo cuando se observa nueva evidencia:

1. `P(Robo=Sí | John llama=Sí, María llama=Sí)`
2. `P(Robo=Sí | Alarma=Sí, Terremoto=No)`

La primera consulta usa como evidencia las llamadas de los vecinos. La segunda consulta usa una evidencia más directa: la alarma sonó y además se descarta el terremoto.


In [2]:
P_B = {True: 0.001, False: 0.999}
P_E = {True: 0.002, False: 0.998}
P_A_TRUE = {
    (True, True): 0.950,
    (True, False): 0.940,
    (False, True): 0.290,
    (False, False): 0.001,
}
P_J_TRUE = {True: 0.90, False: 0.05}
P_M_TRUE = {True: 0.70, False: 0.01}

def p_condicional(valor, prob_true):
    return prob_true if valor else 1 - prob_true

def joint(b, e, a, j, m):
    return (
        p_condicional(b, P_B[True])
        * p_condicional(e, P_E[True])
        * p_condicional(a, P_A_TRUE[(b, e)])
        * p_condicional(j, P_J_TRUE[a])
        * p_condicional(m, P_M_TRUE[a])
    )

num_1 = sum(joint(True, e, a, True, True) for e, a in product([True, False], repeat=2))
den_1 = sum(joint(b, e, a, True, True) for b, e, a in product([True, False], repeat=3))
p_robo_dado_dos_llamadas = num_1 / den_1

num_2 = P_A_TRUE[(True, False)] * P_B[True]
den_2 = num_2 + P_A_TRUE[(False, False)] * P_B[False]
p_robo_dado_alarma_sin_terremoto = num_2 / den_2

resultados_red = pd.DataFrame({
    'Ejemplo': ['1', '2'],
    'Consulta': [
        'P(Robo=Sí | John llama=Sí, María llama=Sí)',
        'P(Robo=Sí | Alarma=Sí, Terremoto=No)',
    ],
    'Probabilidad': [p_robo_dado_dos_llamadas, p_robo_dado_alarma_sin_terremoto],
    'Porcentaje': [p_robo_dado_dos_llamadas * 100, p_robo_dado_alarma_sin_terremoto * 100],
}).round({'Probabilidad': 4, 'Porcentaje': 2})

display(resultados_red)


,Ejemplo,Consulta,Probabilidad,Porcentaje
0,1,"P(Robo=Sí | John llama=Sí, María llama=Sí)",0.2842,28.42
1,2,"P(Robo=Sí | Alarma=Sí, Terremoto=No)",0.4848,48.48


**Interpretación de las predicciones:**

- Si John y María llaman, la probabilidad de robo aumenta de 0.1% a aproximadamente 28.42%. Esto ocurre porque dos evidencias coherentes con la alarma elevan la sospecha de robo, aunque todavía existe la posibilidad de falsas alarmas.
- Si la alarma sonó y se sabe que no hubo terremoto, la probabilidad de robo aumenta a aproximadamente 48.48%. En este caso, al descartar el terremoto, el robo gana peso como explicación probable de la alarma.
- La Red Bayesiana no confirma el evento al 100%, pero permite tomar decisiones con una probabilidad actualizada y justificada por la evidencia disponible.


<div style="background-color:#0a0a0a;color:#00ff41;padding:10px 15px;font-family:'Courier New',Courier,monospace;border-left:5px solid #00ff41;border-bottom:1px solid #00ff41;box-shadow:inset 0 0 5px #000000;">
    <h2 style="margin:0;text-transform:uppercase;">&gt; 2. Fase de preparación de datos_ &#9608;</h2>
</div>

Se usa un dataset nuevo de interés: **Breast Cancer Wisconsin Diagnostic**. La copia local está guardada en:

`data/breast_cancer_wisconsin_diagnostic.csv`

| Elemento | Descripción |
|---|---|
| Registros | 569 |
| Variables predictoras | 30 |
| Tipo de variables | Numéricas continuas |
| Variable objetivo | Diagnóstico |
| Clases | `malignant`, `benign` |

El dataset es adecuado para esta práctica porque plantea un problema de clasificación supervisada: a partir de mediciones numéricas de células tumorales, el modelo debe predecir si el diagnóstico corresponde a una clase maligna o benigna.

En esta fase no se entrena todavía el modelo. Primero se revisa que los datos estén completos, que las variables estén bien separadas y que la clase objetivo esté lista para usarse.


In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

pd.set_option('display.max_columns', 12)

dataset_path = DATA_DIR / 'breast_cancer_wisconsin_diagnostic.csv'
df = pd.read_csv(dataset_path)

X = df.drop(columns=['target', 'diagnostico'])
y = df['target']

info_dataset = pd.DataFrame({
    'metrica': ['archivo', 'filas', 'variables_predictoras', 'valores_faltantes', 'filas_duplicadas', 'clases'],
    'valor': [str(dataset_path), X.shape[0], X.shape[1], X.isna().sum().sum(), df.duplicated().sum(), ', '.join(TARGET_NAMES)],
})

print('Revision inicial del dataset')
display(info_dataset)

print('Primeras 5 filas del dataset')
display(df.head())


Revision inicial del dataset


,metrica,valor
0,archivo,..\data\breast_cancer_wisconsin_diagnostic.csv
1,filas,569
2,variables_predictoras,30
3,valores_faltantes,0
4,filas_duplicadas,0
5,clases,"malignant, benign"


Primeras 5 filas del dataset


,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,...,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target,diagnostico
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,...,0.7119,0.2654,0.4601,0.11890,0,malignant
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,...,0.2416,0.1860,0.2750,0.08902,0,malignant
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,...,0.4504,0.2430,0.3613,0.08758,0,malignant
3,11.42,20.38,77.58,386.1,0.14250,0.28390,...,0.6869,0.2575,0.6638,0.17300,0,malignant
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,...,0.4000,0.1625,0.2364,0.07678,0,malignant


In [4]:
distribucion_clases = df['diagnostico'].value_counts().rename_axis('diagnostico').reset_index(name='cantidad')
distribucion_clases['porcentaje'] = (distribucion_clases['cantidad'] / len(df) * 100).round(2)

display(distribucion_clases)


,diagnostico,cantidad,porcentaje
0,benign,357,62.74
1,malignant,212,37.26


<div style="background-color:#0a0a0a;color:#00ff41;padding:10px 15px;font-family:'Courier New',Courier,monospace;border-left:5px solid #00ff41;border-bottom:1px solid #00ff41;box-shadow:inset 0 0 5px #000000;">
    <h3 style="margin:0;text-transform:uppercase;">&gt; 2.a Preparación aplicada_ &#9608;</h3>
</div>

| Paso | Acción | Resultado |
|---|---|---|
| 1 | Cargar dataset | Se carga desde `data/breast_cancer_wisconsin_diagnostic.csv`. |
| 2 | Revisar calidad | No existen valores faltantes ni duplicados. |
| 3 | Separar variables | `X` contiene 30 mediciones; `y` contiene la clase objetivo. |
| 4 | Dividir datos | 80% entrenamiento y 20% prueba. |
| 5 | Estratificar | Se conserva la proporción de clases. |
| 6 | Preparar pipeline | `StandardScaler` + `GaussianNB`. |

La preparación se realizó antes del entrenamiento para evitar errores comunes como entrenar con columnas objetivo, mezclar clases de forma desbalanceada o evaluar el modelo con datos que ya fueron usados para aprender.

Con esta división, el modelo aprende con el conjunto de entrenamiento y se comprueba con el conjunto de prueba. Esto permite medir si el modelo generaliza a datos que no vio durante el aprendizaje.


In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

split_df = pd.DataFrame({
    'conjunto': ['entrenamiento', 'prueba'],
    'filas': [X_train.shape[0], X_test.shape[0]],
    'variables': [X_train.shape[1], X_test.shape[1]],
})

display(split_df)


,conjunto,filas,variables
0,entrenamiento,455,30
1,prueba,114,30


<div style="background-color:#0a0a0a;color:#00ff41;padding:10px 15px;font-family:'Courier New',Courier,monospace;border-left:5px solid #00ff41;border-bottom:1px solid #00ff41;box-shadow:inset 0 0 5px #000000;">
    <h2 style="margin:0;text-transform:uppercase;">&gt; 3. Fase de modelado: Modelo Naive Bayes_ &#9608;</h2>
</div>

Se desarrolla un modelo **Gaussian Naive Bayes**, adecuado para variables numéricas continuas. Este algoritmo aplica el razonamiento bayesiano suponiendo independencia condicional entre las variables predictoras dada la clase.

**Diagrama del proceso de modelado:**

![Flujo profesional de modelado Naive Bayes](../img/flujo_naive_bayes.svg)

| Etapa | Descripción |
|---|---|
| Entrada | 30 variables numéricas del dataset. |
| Escalamiento | `StandardScaler` normaliza la escala de las variables. |
| Modelo | `GaussianNB` calcula probabilidades por clase. |
| Salida | Clase predicha: `malignant` o `benign`. |
| Evaluación | Accuracy, reporte de clasificación y matriz de confusión. |

Aunque la independencia total entre variables no siempre se cumple en datos reales, Naive Bayes es una excelente línea base porque permite obtener predicciones rápidas, interpretables y expresadas como probabilidades.

En términos sencillos, el modelo compara qué tan compatible es un nuevo registro con la clase `malignant` y con la clase `benign`; luego selecciona la clase con mayor probabilidad.


In [6]:
modelo_nb = Pipeline([
    ('scaler', StandardScaler()),
    ('naive_bayes', GaussianNB()),
])

modelo_nb.fit(X_train, y_train)
y_pred = modelo_nb.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)

reporte_df = (
    pd.DataFrame(classification_report(y_test, y_pred, target_names=TARGET_NAMES, output_dict=True))
    .T
    .reset_index()
    .rename(columns={'index': 'clase/metrica'})
    .round(4)
)

display(reporte_df)


,clase/metrica,precision,recall,f1-score,support
0,malignant,0.9048,0.9048,0.9048,42.0000
1,benign,0.9444,0.9444,0.9444,72.0000
2,accuracy,0.9298,0.9298,0.9298,0.9298
3,macro avg,0.9246,0.9246,0.9246,114.0000
4,weighted avg,0.9298,0.9298,0.9298,114.0000


In [7]:
matriz_df = pd.DataFrame(
    confusion_matrix(y_test, y_pred),
    index=TARGET_NAMES,
    columns=TARGET_NAMES,
).rename_axis(index='Clase real', columns='Prediccion')

metricas_modelo = pd.DataFrame({
    'metrica': ['Accuracy', 'Errores en prueba', 'Total muestras prueba'],
    'valor': [round(accuracy, 4), int((y_test != y_pred).sum()), len(y_test)],
})

print('Metricas generales del modelo')
display(metricas_modelo)

print('Matriz de confusion')
display(matriz_df)


Metricas generales del modelo


,metrica,valor
0,Accuracy,0.9298
1,Errores en prueba,8.0000
2,Total muestras prueba,114.0000


Matriz de confusion


Prediccion,malignant,benign
Clase real,,
malignant,38,4
benign,4,68


<div style="background-color:#0a0a0a;color:#00ff41;padding:10px 15px;font-family:'Courier New',Courier,monospace;border-left:5px solid #00ff41;border-bottom:1px solid #00ff41;box-shadow:inset 0 0 5px #000000;">
    <h2 style="margin:0;text-transform:uppercase;">&gt; 4. Fase de predicción de nuevos samples_ &#9608;</h2>
</div>

Se realiza la predicción con dos nuevos samples guardados en:

`data/nuevos_samples_bayes.csv`

| Sample | Descripción |
|---|---|
| `sample_nuevo_perfil_benigno` | Perfil construido con medianas de tumores benignos. |
| `sample_nuevo_perfil_maligno` | Perfil construido con medianas de tumores malignos. |

Estos samples no se toman directamente como filas del dataset original; se construyen como perfiles representativos para comprobar si el modelo responde de forma coherente ante nuevos casos.

La tabla final muestra la clase predicha y la probabilidad asignada a cada clase. Esto es importante porque no solo indica una respuesta, sino también el nivel de confianza del modelo.


In [8]:
samples_path = DATA_DIR / 'nuevos_samples_bayes.csv'
nuevos_samples = pd.read_csv(samples_path).set_index('sample')

display(nuevos_samples.iloc[:, :10].reset_index().round(4))


,sample,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension
0,sample_nuevo_perfil_benigno,12.200,17.39,78.18,458.4,0.0908,0.0753,0.0371,0.0234,0.1714,0.0615
1,sample_nuevo_perfil_maligno,17.325,21.46,114.20,932.0,0.1022,0.1324,0.1513,0.0863,0.1899,0.0616


In [9]:
predicciones = modelo_nb.predict(nuevos_samples)
probabilidades = modelo_nb.predict_proba(nuevos_samples)

resultado_samples = pd.DataFrame({
    'sample': nuevos_samples.index,
    'prediccion_num': predicciones,
    'prediccion_clase': [TARGET_NAMES[i] for i in predicciones],
    'P_malignant': probabilidades[:, 0],
    'P_benign': probabilidades[:, 1],
}).round({'P_malignant': 4, 'P_benign': 4})

display(resultado_samples)


,sample,prediccion_num,prediccion_clase,P_malignant,P_benign
0,sample_nuevo_perfil_benigno,1,benign,0.0,1.0
1,sample_nuevo_perfil_maligno,0,malignant,1.0,0.0


<div style="background-color:#0a0a0a;color:#00ff41;padding:10px 15px;font-family:'Courier New',Courier,monospace;border-left:5px solid #00ff41;border-bottom:1px solid #00ff41;box-shadow:inset 0 0 5px #000000;">
    <h3 style="margin:0;text-transform:uppercase;">&gt; 4.a Justificación del modelo elegido_ &#9608;</h3>
</div>

Se eligió `GaussianNB` porque el dataset contiene variables continuas relacionadas con mediciones geométricas y estadísticas. El modelo es una línea base adecuada porque:

- Es rápido de entrenar.
- Entrega probabilidades por clase.
- Es fácil de explicar en una práctica de Bayes.
- Se conecta directamente con el objetivo de soporte a decisiones.

La elección del modelo también es coherente con el enfoque académico de la práctica: no solo se busca obtener una predicción, sino entender cómo una decisión puede apoyarse en probabilidades calculadas a partir de datos.


<div style="background-color:#0a0a0a;color:#00ff41;padding:10px 15px;font-family:'Courier New',Courier,monospace;border-left:5px solid #00ff41;border-bottom:1px solid #00ff41;box-shadow:inset 0 0 5px #000000;">
    <h2 style="margin:0;text-transform:uppercase;">&gt; 7. Conclusiones y referencias APA_ &#9608;</h2>
</div>

- El teorema de Bayes permite actualizar probabilidades iniciales cuando aparece nueva evidencia, lo cual resulta fundamental para resolver problemas donde no existe certeza absoluta.
- Las Redes Bayesianas representan dependencias entre variables y permiten razonar con incertidumbre de forma estructurada. En la red de alarma, las llamadas de John y María aumentan la probabilidad de robo, pero no la confirman al 100%.
- La fase de preparación dejó un dataset nuevo, limpio y listo para clasificación, con variables predictoras separadas correctamente de la variable objetivo.
- El modelo `GaussianNB` alcanzó un accuracy aproximado de 92.98% en prueba, por lo que demuestra un desempeño sólido como línea base para clasificación probabilística.
- La predicción de dos nuevos samples permitió comprobar que el modelo no solo clasifica, sino que también entrega probabilidades por clase, lo que fortalece su utilidad como herramienta inicial de apoyo a decisiones.
- Como mejora futura, se podría comparar Naive Bayes con otros algoritmos de clasificación y analizar si la correlación entre variables afecta el supuesto de independencia del modelo.


<div style="background-color:#0a0a0a;color:#00ff41;padding:10px 15px;font-family:'Courier New',Courier,monospace;border-left:5px solid #00ff41;border-bottom:1px solid #00ff41;box-shadow:inset 0 0 5px #000000;">
    <h3 style="margin:0;text-transform:uppercase;">&gt; Referencias APA_ &#9608;</h3>
</div>

Russell, S., & Norvig, P. (2021). *Artificial intelligence: A modern approach* (4th ed.). Pearson. https://aima.cs.berkeley.edu/

Scikit-learn developers. (2026). *Naive Bayes*. Scikit-learn documentation. https://scikit-learn.org/stable/modules/naive_bayes.html

Scikit-learn developers. (2026). *load_breast_cancer*. Scikit-learn documentation. https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html

Wolberg, W., Mangasarian, O., & Street, N. (1993). *Breast Cancer Wisconsin (Diagnostic)* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5DW2B

Zhang, H. (2004). The optimality of naive Bayes. *Proceedings of the Seventeenth International Florida Artificial Intelligence Research Society Conference*. https://www.cs.unb.ca/~hzhang/publications/FLAIRS04ZhangH.pdf
